# Diagnostic J1
Ce notebook sert à diagnostiquer les fichiers FAO et à vérifier les importations, les chemins et les premières analyses.

## Objectifs
- Charger les données depuis le dossier `data/`
- Vérifier la structure des fichiers CSV
- Nettoyer les valeurs économiques impossibles
- Vérifier la méthode de jointure `inner` sur `Code zone`
- Identifier les codes pays exclus par les jointures

In [ ]:
import pandas as pd
from profiler import ProfileurFAO
from pathlib import Path

# Chemin des données

data_dir = Path("data")
print(f"Dossier des données : {data_dir.resolve()}")
print(f"Existe : {data_dir.exists()}")

In [ ]:
files = sorted(data_dir.glob("fr_*.csv"))
print("Fichiers trouvés :")
for f in files:
    print(f"- {f.name}")

In [ ]:
if files:
    df = pd.read_csv(files[0])
    print(df.head())
    print(df.columns.tolist())
else:
    print("Aucun fichier CSV trouvé dans data/")

## Nettoyage des valeurs négatives pour les éléments de type `Production`

Les valeurs négatives sur les éléments de type `Production` ont été remplacées par `NaN`, car elles ne sont pas interprétables économiquement. En effet, une production négative n'est pas possible : il s'agit probablement d'une erreur de saisie ou d'un code spécial.

Le choix de conserver la période la plus récente et cohérente avec les autres données (2013) permet de garantir une comparaison stable entre séries temporelles. Cela évite de conserver des valeurs aberrantes qui fausseraient les agrégats et les analyses de tendance.

In [ ]:
# Chargement des principaux fichiers pour le diagnostic
files_to_load = [
    "fr_vegetaux.csv",
    "fr_animaux.csv",
    "fr_cereales.csv"
]

dfs = {}
for name in files_to_load:
    path = data_dir / name
    dfs[name] = pd.read_csv(path)
    print(f"Chargé {name} -> shape {dfs[name].shape}")

# Fonction de nettoyage

def clean_negative_production(df):
    df = df.copy()
    if "Élément" in df.columns:
        mask = df["Élément"] == "Production"
    elif "Element" in df.columns:
        mask = df["Element"] == "Production"
    else:
        return df

    if "Valeur" in df.columns:
        df.loc[mask & (pd.to_numeric(df["Valeur"], errors="coerce") < 0), "Valeur"] = pd.NA
    return df

for name, df in dfs.items():
    before = pd.to_numeric(df["Valeur"], errors="coerce").lt(0).sum()
    dfs[name] = clean_negative_production(df)
    after = pd.to_numeric(dfs[name]["Valeur"], errors="coerce").lt(0).sum()
    print(f"{name}: négatives avant={before}, après={after}")

In [ ]:
# Vérifier la période la plus récente et la cohérence autour de 2013
for name, df in dfs.items():
    if "Code année" in df.columns:
        years = sorted(df["Code année"].dropna().unique())
        print(f"{name} : années min={years[0]} max={years[-1]} (extrait) -> {years[:5]} ... {years[-5:]}")
    else:
        print(f"{name} : pas de colonne Code année")

## Jointure des données

Méthode de jointure : `inner` sur `Code zone` → on garde uniquement les pays présents dans les trois fichiers.

Cette méthode garantit que la table finale ne contient que les pays communs à tous les jeux de données, ce qui est essentiel pour des comparaisons homogènes et des analyses multi-sources.

In [ ]:
# Préparer les DataFrames pour la jointure
key = "Code zone"
common_codes = None
for name, df in dfs.items():
    codes = set(df[key].dropna().unique())
    print(f"{name} : {len(codes)} codes uniques")
    if common_codes is None:
        common_codes = codes
    else:
        common_codes = common_codes.intersection(codes)

print(f"Codes communs aux trois fichiers : {len(common_codes)}")

# Exemple de jointure inner entre deux fichiers puis troisième
joined_1_2 = dfs["fr_vegetaux.csv"].merge(
    dfs["fr_animaux.csv"], on=key, how="inner", suffixes=("_veg", "_anim")
)
joined_all = joined_1_2.merge(
    dfs["fr_cereales.csv"], on=key, how="inner"
)
print(f"Shape après jointure inner sur les trois fichiers : {joined_all.shape}")

In [ ]:
# Pays exclus : codes présents dans un fichier mais absents dans un autre
code_sets = {name: set(df[key].dropna().unique()) for name, df in dfs.items()}

for name_a, set_a in code_sets.items():
    for name_b, set_b in code_sets.items():
        if name_a != name_b:
            missing = sorted(set_a - set_b)
            print(f"Codes dans {name_a} absents de {name_b} : {len(missing)}")
            if len(missing) <= 20:
                print(missing)
            else:
                print(missing[:20], "...")
    print("---")

## Conclusions intermédiaires
- Le nettoyage remplace les `Production` négatives par `NaN` afin d'éviter des valeurs économiquement impossibles.
- La jointure `inner` sur `Code zone` conserve uniquement les pays présents dans les trois jeux.
- Les codes exclus sont listés par comparaison de `set` entre fichiers.
- Ce diagnostic permet de vérifier que les jeux sont cohérents avant la suite du traitement.